# Week 11: Frameworks (LangChain & LlamaIndex)

**Lab companion to [week_11.md](../agenda/week_11.md)**

In this lab, you will:
1. Use LangChain for chains and agents
2. Use LlamaIndex for RAG
3. Compare frameworks with raw OpenAI
4. Build practical examples

In [ ]:
# Install frameworks
!pip install langchain langchain-openai llama-index -q

In [ ]:
from dotenv import load_dotenv
load_dotenv()
print("✓ Ready!")

## 1. LangChain Basics

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Create LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Simple call
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is Python?")
]

response = llm.invoke(messages)
print(response.content)

In [ ]:
# Using prompt templates
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert on {topic}. Be concise."),
    ("human", "{question}")
])

# Create chain
chain = prompt | llm

# Run
result = chain.invoke({
    "topic": "machine learning",
    "question": "What is gradient descent?"
})

print(result.content)

In [ ]:
# Output parsers
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel

class Topic(BaseModel):
    name: str
    description: str
    difficulty: str

parser = JsonOutputParser(pydantic_object=Topic)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract topic information. {format_instructions}"),
    ("human", "Tell me about: {topic}")
])

chain = prompt | llm | parser

result = chain.invoke({
    "topic": "neural networks",
    "format_instructions": parser.get_format_instructions()
})

print(result)

## 2. LangChain Tools & Agents

In [ ]:
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """Calculate a math expression. Input: math expression like '2 + 2'."""
    try:
        return str(eval(expression))
    except:
        return "Error"

@tool  
def search(query: str) -> str:
    """Search for information. Input: search query."""
    return f"Mock results for: {query}"

tools = [calculate, search]
print(f"Defined {len(tools)} tools")

In [ ]:
# Create agent with tools
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools when needed."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run
result = agent_executor.invoke({"input": "What is 15 * 7?"})
print(f"\nAnswer: {result['output']}")

## 3. LlamaIndex for RAG

In [ ]:
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure
Settings.llm = LlamaOpenAI(model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Create documents
documents = [
    Document(text="Python is a programming language created by Guido van Rossum in 1991."),
    Document(text="Machine learning uses algorithms to learn patterns from data."),
    Document(text="RAG combines retrieval with generation for better AI responses."),
    Document(text="Vector databases store embeddings for semantic search.")
]

# Create index
index = VectorStoreIndex.from_documents(documents)

print(f"✓ Indexed {len(documents)} documents")

In [ ]:
# Query the index
query_engine = index.as_query_engine()

response = query_engine.query("What is RAG?")
print(response)

In [ ]:
# Chat engine with memory
chat_engine = index.as_chat_engine(chat_mode="condense_question")

response1 = chat_engine.chat("Tell me about Python")
print(f"Q1: {response1}\n")

response2 = chat_engine.chat("Who created it?")
print(f"Q2: {response2}")

## 4. Framework Comparison

In [ ]:
# Raw OpenAI
from openai import OpenAI

raw_client = OpenAI()

def raw_openai_query(question: str) -> str:
    response = raw_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# LangChain
def langchain_query(question: str) -> str:
    return llm.invoke(question).content

# LlamaIndex (with RAG)
def llamaindex_query(question: str) -> str:
    return str(query_engine.query(question))

# Compare
question = "What is machine learning?"

print("Raw OpenAI:")
print(raw_openai_query(question)[:200] + "...\n")

print("LangChain:")
print(langchain_query(question)[:200] + "...\n")

print("LlamaIndex (RAG):")
print(llamaindex_query(question))

## 5. When to Use What?

| Scenario | Recommendation |
|----------|---------------|
| Simple API calls | Raw OpenAI |
| Complex chains | LangChain |
| Document Q&A | LlamaIndex |
| Tool-using agents | LangChain |
| Production RAG | Either, or raw |
| Learning/prototyping | Frameworks |
| Maximum control | Raw OpenAI |

## Summary

You learned:
- ✅ LangChain: prompts, chains, agents
- ✅ LlamaIndex: indexing, querying, chat
- ✅ Comparing frameworks
- ✅ When to use which approach

**Next:** [Week 12 - Multi-modal](week_12_multimodal.ipynb)